# Bài Toán RAG Fine-tuning với Qwen2.5-7B-Instruct
Đây là quy trình chuẩn 6 bước sử dụng Unsloth trên Google Colab T4 16GB VRAM.
Hãy upload file `dataset.jsonl` của bạn lên chung thư mục với file này nhé!

### Bước 1: Cài đặt công cụ (Install Dependencies)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

### Bước 2: Nạp mô hình 4-bit (Load Model)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Độ dài context window
dtype = None
load_in_4bit = True # Bắt buộc phải = True trên T4 16GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Bước 3: Cấu hình dán giấy nhớ (Setup LoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Kích thước giấy nhớ (đề xuất: 8, 16, 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Bước 4: Biến hình Dữ liệu (Data Formatting)
Đọc file `dataset.jsonl` và bọc lại theo chuẩn ChatML.

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# 1. Đọc file JSONL của bạn
def load_dataset(jsonl_path):
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return Dataset.from_list(data)

# Nhớ upload file dataset.jsonl lên Colab nhé!
dataset = load_dataset("dataset.jsonl")

# 2. Dùng chuẩn Chat Template của Qwen 2.5
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# 3. Format lại data
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample data:", dataset[0]["text"][:500])

### Bước 5: Bắt đầu Huấn luyện (Training)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Chạy 1 epoch (duyệt qua toàn bộ dataset 1 lần)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Bấm nút chạy và đi uống cà phê!
trainer_stats = trainer.train()

### Bước 6: Xuất xưởng và Tải về (Export to GGUF)

In [ ]:
# Export ra định dạng q4_k_m cực kì nhẹ (khoảng ~5GB)
# File này tải về chạy trực tiếp trên LM Studio hoặc Ollama không cần code!
model.save_pretrained_gguf("qwen2.5-7b-rag-finetuned", tokenizer, quantization_method = "q4_k_m")

# Nếu bạn có tài khoản Hugging Face, bỏ comment dòng dưới để up thẳng lên mạng (Nhớ đổi tên username)
# model.push_to_hub_gguf("YOUR_HF_USERNAME/qwen2.5-rag", tokenizer, token = "hf_...")